# End-to-end YOLO11n shoe-brand classification

Train, tune, evaluate, analyze, and export a `yolo11n-cls.pt` model for Adidas, Converse, and Nike. Run the cells sequentially after completing dataset setup, stratified splitting, preprocessing, and EDA.

## 0. Install dependencies into the active kernel

This cell installs packages through the exact Python executable used by this notebook. Restart the kernel if Jupyter reports that newly installed native packages cannot be loaded, then continue from the import cell.

In [ ]:
import subprocess
import sys

REQUIRED_PACKAGES = [
    "matplotlib",
    "numpy",
    "pandas",
    "Pillow",
    "PyYAML",
    "scikit-learn",
    "seaborn",
    "torch",
    "torchvision",
    "ultralytics==8.4.213",
]
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    *REQUIRED_PACKAGES,
])
print(f"Dependencies installed for kernel: {sys.executable}")

## 1. Imports and experiment configuration

In [1]:
import json
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from ultralytics import YOLO, __version__ as ultralytics_version

DATASET_ROOT = Path("../datasets/shoe_brands_preprocessed").resolve()
RUNS_ROOT = Path("../runs/training").resolve()
EVALUATION_ROOT = RUNS_ROOT / "evaluation"
MODEL_CHECKPOINT = "yolo11n-cls.pt"
EXPECTED_CLASSES = ["adidas", "converse", "nike"]
EXPECTED_COUNTS = {"train": 193, "val": 55, "test": 27}
IMAGE_SIZE = 224
BATCH_SIZE = 32
WORKERS = min(8, os.cpu_count() or 1)
SEED = 37
BASELINE_EPOCHS = 30
TUNING_ITERATIONS = 20
TUNING_EPOCHS = 20
FINAL_EPOCHS = 100
PATIENCE = 20
RUN_TUNING = True
EXPORT_ONNX = True
RESUME_TRAINING = False

ModuleNotFoundError: No module named 'pandas'

## 2. Runtime, device, and reproducibility

In [ ]:
def select_device() -> str | int:
    if torch.cuda.is_available():
        return 0
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = select_device()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Python executable: {os.sys.executable}")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics_version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Dataset validation

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
dataset_counts = {}
for split_name, expected_count in EXPECTED_COUNTS.items():
    split_path = DATASET_ROOT / split_name
    if not split_path.is_dir():
        raise FileNotFoundError(f"Missing dataset split: {split_path}")

    actual_classes = sorted(path.name for path in split_path.iterdir() if path.is_dir())
    if actual_classes != EXPECTED_CLASSES:
        raise ValueError(f"Unexpected classes in {split_name}: {actual_classes}")

    dataset_counts[split_name] = {}
    for class_name in EXPECTED_CLASSES:
        image_count = sum(
            1 for path in (split_path / class_name).iterdir()
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
        )
        dataset_counts[split_name][class_name] = image_count
        if image_count != expected_count:
            raise ValueError(
                f"Expected {expected_count} images in {split_name}/{class_name}; found {image_count}."
            )

dataset_distribution = pd.DataFrame(dataset_counts).T[EXPECTED_CLASSES]
dataset_distribution["total"] = dataset_distribution.sum(axis=1)
display(dataset_distribution)
print(f"Dataset ready: {DATASET_ROOT}")

## 4. Baseline training

In [ ]:
baseline_model = YOLO(MODEL_CHECKPOINT)
baseline_started_at = time.perf_counter()
baseline_results = baseline_model.train(
    data=str(DATASET_ROOT),
    epochs=BASELINE_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=3,
    dropout=0.1,
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=8.0,
    translate=0.08,
    scale=0.2,
    fliplr=0.5,
    flipud=0.0,
    cos_lr=True,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    cache=False,
    save=True,
    save_period=10,
    project=str(RUNS_ROOT),
    name="baseline",
    exist_ok=True,
    plots=True,
)
baseline_duration_seconds = time.perf_counter() - baseline_started_at
BASELINE_RUN_DIR = Path(baseline_results.save_dir)
BASELINE_BEST_PATH = BASELINE_RUN_DIR / "weights" / "best.pt"
print(f"Baseline duration: {baseline_duration_seconds / 60:.2f} minutes")
print(f"Baseline best checkpoint: {BASELINE_BEST_PATH}")

## 5. Training curves

In [ ]:
def load_training_history(run_directory: Path) -> pd.DataFrame:
    results_path = run_directory / "results.csv"
    if not results_path.is_file():
        raise FileNotFoundError(f"Training history not found: {results_path}")
    history = pd.read_csv(results_path)
    history.columns = history.columns.str.strip()
    return history


def plot_training_history(history: pd.DataFrame, title: str) -> None:
    metric_columns = [
        column for column in history.columns
        if any(keyword in column.lower() for keyword in ("loss", "accuracy", "top1", "top5", "lr/"))
    ]
    if not metric_columns:
        print(f"No recognized metric columns. Available columns: {list(history.columns)}")
        return

    columns = 2
    rows = int(np.ceil(len(metric_columns) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(14, 4 * rows), squeeze=False)
    for axis, metric_name in zip(axes.flat, metric_columns):
        axis.plot(history.index + 1, history[metric_name])
        axis.set_title(metric_name)
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.3)
    for axis in axes.flat[len(metric_columns):]:
        axis.axis("off")
    figure.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


baseline_history = load_training_history(BASELINE_RUN_DIR)
plot_training_history(baseline_history, "Baseline training curves")

## 6. Hyperparameter tuning

In [ ]:
TUNING_PROJECT = RUNS_ROOT / "tuning"
TUNING_SPACE = {
    "lr0": (1e-5, 1e-2),
    "lrf": (0.01, 0.2),
    "momentum": (0.7, 0.98),
    "weight_decay": (0.0, 0.001),
    "warmup_epochs": (0.0, 5.0),
    "hsv_h": (0.0, 0.03),
    "hsv_s": (0.0, 0.5),
    "hsv_v": (0.0, 0.4),
    "degrees": (0.0, 12.0),
    "translate": (0.0, 0.12),
    "scale": (0.0, 0.3),
    "fliplr": (0.0, 0.5),
    "dropout": (0.0, 0.3),
}

if RUN_TUNING:
    tuning_model = YOLO(MODEL_CHECKPOINT)
    tuning_model.tune(
        data=str(DATASET_ROOT),
        epochs=TUNING_EPOCHS,
        iterations=TUNING_ITERATIONS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        optimizer="AdamW",
        seed=SEED,
        deterministic=True,
        space=TUNING_SPACE,
        project=str(TUNING_PROJECT),
        name="yolo11n_cls_tune",
        plots=True,
        save=True,
        val=True,
    )

## 7. Load and review best tuning parameters

In [ ]:
best_hyperparameter_files = sorted(
    TUNING_PROJECT.rglob("best_hyperparameters.yaml"),
    key=lambda path: path.stat().st_mtime,
)
if RUN_TUNING and not best_hyperparameter_files:
    raise FileNotFoundError(f"No best_hyperparameters.yaml found below {TUNING_PROJECT}")

if best_hyperparameter_files:
    BEST_HYPERPARAMETERS_PATH = best_hyperparameter_files[-1]
    with BEST_HYPERPARAMETERS_PATH.open("r", encoding="utf-8") as yaml_file:
        tuned_hyperparameters = yaml.safe_load(yaml_file) or {}
else:
    BEST_HYPERPARAMETERS_PATH = None
    tuned_hyperparameters = {}

ALLOWED_TUNED_PARAMETERS = {
    "lr0", "lrf", "momentum", "weight_decay", "warmup_epochs",
    "hsv_h", "hsv_s", "hsv_v", "degrees", "translate",
    "scale", "fliplr", "flipud", "dropout",
}
final_hyperparameters = {
    key: value for key, value in tuned_hyperparameters.items()
    if key in ALLOWED_TUNED_PARAMETERS
}

print(f"Best hyperparameters file: {BEST_HYPERPARAMETERS_PATH}")
display(pd.DataFrame.from_dict(final_hyperparameters, orient="index", columns=["value"]))

In [ ]:
tuning_csv_files = sorted(TUNING_PROJECT.rglob("tune_results.csv"))
if tuning_csv_files:
    tuning_history = pd.read_csv(tuning_csv_files[-1])
    display(tuning_history.sort_values("fitness", ascending=False).head(10))
    if "fitness" in tuning_history.columns:
        tuning_history["fitness"].plot(figsize=(10, 4), marker="o", title="Tuning fitness by iteration")
        plt.xlabel("Iteration")
        plt.ylabel("Fitness")
        plt.grid(alpha=0.3)
        plt.show()
else:
    print("No tune_results.csv found; inspect the tuning directory manually.")

## 8. Final training from fresh pretrained weights

In [ ]:
final_training_arguments = {
    "data": str(DATASET_ROOT),
    "epochs": FINAL_EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "device": DEVICE,
    "workers": WORKERS,
    "optimizer": "AdamW",
    "patience": PATIENCE,
    "cos_lr": True,
    "seed": SEED,
    "deterministic": True,
    "cache": False,
    "save": True,
    "save_period": 10,
    "project": str(RUNS_ROOT),
    "name": "final",
    "exist_ok": True,
    "plots": True,
}
final_training_arguments.update(final_hyperparameters)

final_model = YOLO(MODEL_CHECKPOINT)
final_started_at = time.perf_counter()
final_results = final_model.train(**final_training_arguments)
final_duration_seconds = time.perf_counter() - final_started_at
FINAL_RUN_DIR = Path(final_results.save_dir)
FINAL_BEST_PATH = FINAL_RUN_DIR / "weights" / "best.pt"
FINAL_LAST_PATH = FINAL_RUN_DIR / "weights" / "last.pt"
if not FINAL_BEST_PATH.is_file():
    raise FileNotFoundError(f"Final best checkpoint not found: {FINAL_BEST_PATH}")

print(f"Final training duration: {final_duration_seconds / 60:.2f} minutes")
print(f"Final best checkpoint: {FINAL_BEST_PATH}")

In [ ]:
final_history = load_training_history(FINAL_RUN_DIR)
plot_training_history(final_history, "Final training curves")

## 9. Validation and test evaluation

In [ ]:
best_model = YOLO(str(FINAL_BEST_PATH))
validation_metrics = best_model.val(
    data=str(DATASET_ROOT), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE, plots=True
)
test_metrics = best_model.val(
    data=str(DATASET_ROOT), split="test", imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE, plots=True
)

evaluation_overview = pd.DataFrame({
    "split": ["validation", "test"],
    "top1_accuracy": [validation_metrics.top1, test_metrics.top1],
    "top5_accuracy": [validation_metrics.top5, test_metrics.top5],
})
display(evaluation_overview)
print(f"Validation speed: {validation_metrics.speed}")
print(f"Test speed: {test_metrics.speed}")

## 10. Test-set predictions

In [ ]:
prediction_records = []
test_results = best_model.predict(
    source=str(DATASET_ROOT / "test"),
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    stream=True,
    verbose=False,
)
for result in test_results:
    if result.probs is None:
        continue
    probabilities = result.probs.data.detach().cpu().numpy()
    predicted_index = int(result.probs.top1)
    image_path = Path(result.path)
    record = {
        "image_path": str(image_path),
        "true_label": image_path.parent.name,
        "predicted_label": result.names[predicted_index],
        "confidence": float(result.probs.top1conf.detach().cpu()),
    }
    for class_index, probability in enumerate(probabilities):
        record[f"probability_{result.names[class_index]}"] = float(probability)
    prediction_records.append(record)

predictions = pd.DataFrame(prediction_records)
if len(predictions) != sum(EXPECTED_COUNTS["test"] for _ in EXPECTED_CLASSES):
    raise RuntimeError(f"Expected 81 test predictions; found {len(predictions)}.")
predictions["correct"] = predictions["true_label"] == predictions["predicted_label"]
display(predictions.head())

## 11. Classification metrics and confusion matrices

In [ ]:
true_labels = predictions["true_label"]
predicted_labels = predictions["predicted_label"]
metrics_summary = {
    "accuracy": accuracy_score(true_labels, predicted_labels),
    "macro_precision": precision_score(true_labels, predicted_labels, average="macro", zero_division=0),
    "macro_recall": recall_score(true_labels, predicted_labels, average="macro", zero_division=0),
    "macro_f1": f1_score(true_labels, predicted_labels, average="macro", zero_division=0),
    "weighted_f1": f1_score(true_labels, predicted_labels, average="weighted", zero_division=0),
    "validation_top1": float(validation_metrics.top1),
    "validation_top5": float(validation_metrics.top5),
    "test_top1": float(test_metrics.top1),
    "test_top5": float(test_metrics.top5),
}
display(pd.Series(metrics_summary, name="value").to_frame())

report_dict = classification_report(
    true_labels, predicted_labels, labels=EXPECTED_CLASSES, output_dict=True, zero_division=0
)
classification_report_frame = pd.DataFrame(report_dict).T
display(classification_report_frame)

In [ ]:
confusion_counts = confusion_matrix(true_labels, predicted_labels, labels=EXPECTED_CLASSES)
confusion_normalized = confusion_matrix(
    true_labels, predicted_labels, labels=EXPECTED_CLASSES, normalize="true"
)

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_counts, annot=True, fmt="d", cmap="Blues", xticklabels=EXPECTED_CLASSES, yticklabels=EXPECTED_CLASSES, ax=axes[0])
axes[0].set_title("Test confusion matrix")
sns.heatmap(confusion_normalized, annot=True, fmt=".2f", cmap="Greens", xticklabels=EXPECTED_CLASSES, yticklabels=EXPECTED_CLASSES, ax=axes[1])
axes[1].set_title("Normalized test confusion matrix")
for axis in axes:
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("True label")
plt.tight_layout()
plt.show()

## 12. Error and confidence analysis

In [ ]:
misclassifications = predictions[~predictions["correct"]].sort_values("confidence", ascending=False)
display(misclassifications)

figure, axis = plt.subplots(figsize=(9, 4))
sns.histplot(
    data=predictions, x="confidence", hue="correct", bins=15, multiple="layer",
    stat="count", common_norm=False, ax=axis,
)
axis.set_title("Confidence distribution: correct vs incorrect predictions")
plt.show()

display(predictions.nsmallest(10, "confidence"))

In [ ]:
def show_prediction_grid(rows: pd.DataFrame, title: str, max_images: int = 12) -> None:
    rows = rows.head(max_images)
    if rows.empty:
        print(f"No images to display for {title}.")
        return
    columns = 3
    row_count = int(np.ceil(len(rows) / columns))
    figure, axes = plt.subplots(row_count, columns, figsize=(12, 4 * row_count), squeeze=False)
    for axis, (_, row) in zip(axes.flat, rows.iterrows()):
        with Image.open(row["image_path"]) as image:
            axis.imshow(image)
        axis.set_title(
            f"True: {row['true_label']}\nPred: {row['predicted_label']} ({row['confidence']:.1%})"
        )
        axis.axis("off")
    for axis in axes.flat[len(rows):]:
        axis.axis("off")
    figure.suptitle(title)
    plt.tight_layout()
    plt.show()


show_prediction_grid(misclassifications, "Highest-confidence misclassifications")

## 13. Baseline versus final model

In [ ]:
baseline_best_model = YOLO(str(BASELINE_BEST_PATH))
baseline_validation_metrics = baseline_best_model.val(
    data=str(DATASET_ROOT), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE
)
model_comparison = pd.DataFrame([
    {
        "model": "baseline",
        "validation_top1": float(baseline_validation_metrics.top1),
        "validation_top5": float(baseline_validation_metrics.top5),
        "training_minutes": baseline_duration_seconds / 60,
        "checkpoint": str(BASELINE_BEST_PATH),
    },
    {
        "model": "final_tuned",
        "validation_top1": float(validation_metrics.top1),
        "validation_top5": float(validation_metrics.top5),
        "training_minutes": final_duration_seconds / 60,
        "checkpoint": str(FINAL_BEST_PATH),
    },
])
display(model_comparison)

## 14. Save evaluation artifacts

In [ ]:
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
predictions.to_csv(EVALUATION_ROOT / "test_predictions.csv", index=False)
misclassifications.to_csv(EVALUATION_ROOT / "misclassifications.csv", index=False)
classification_report_frame.to_csv(EVALUATION_ROOT / "classification_report.csv")
pd.DataFrame(confusion_counts, index=EXPECTED_CLASSES, columns=EXPECTED_CLASSES).to_csv(
    EVALUATION_ROOT / "confusion_matrix.csv"
)
model_comparison.to_csv(EVALUATION_ROOT / "model_comparison.csv", index=False)
with (EVALUATION_ROOT / "metrics_summary.json").open("w", encoding="utf-8") as metrics_file:
    json.dump(metrics_summary, metrics_file, indent=2)

experiment_manifest = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "dataset_root": str(DATASET_ROOT),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "device": str(DEVICE),
    "ultralytics_version": ultralytics_version,
    "pytorch_version": torch.__version__,
    "best_checkpoint": str(FINAL_BEST_PATH),
    "best_hyperparameters": final_hyperparameters,
}
with (EVALUATION_ROOT / "experiment_manifest.json").open("w", encoding="utf-8") as manifest_file:
    json.dump(experiment_manifest, manifest_file, indent=2, default=str)
print(f"Evaluation artifacts saved to: {EVALUATION_ROOT}")

## 15. Reusable single-image inference

In [ ]:
def classify_shoe(image_source: str | Path) -> dict:
    result = best_model.predict(
        source=str(image_source), imgsz=IMAGE_SIZE, device=DEVICE, verbose=False
    )[0]
    if result.probs is None:
        raise RuntimeError("The model returned no classification probabilities.")
    probabilities = result.probs.data.detach().cpu().numpy()
    predicted_index = int(result.probs.top1)
    return {
        "class_name": result.names[predicted_index],
        "confidence": float(result.probs.top1conf.detach().cpu()),
        "probabilities": {
            result.names[index]: float(probability)
            for index, probability in enumerate(probabilities)
        },
    }


sample_image = Path(predictions.iloc[0]["image_path"])
print(classify_shoe(sample_image))

## 16. Export the best model to ONNX

In [ ]:
if EXPORT_ONNX:
    exported_model_path = best_model.export(
        format="onnx",
        imgsz=IMAGE_SIZE,
        batch=1,
        dynamic=False,
        simplify=True,
        device=DEVICE,
    )
    print(f"Exported ONNX model: {exported_model_path}")

## 17. Optional resume training

In [ ]:
if RESUME_TRAINING:
    if not FINAL_LAST_PATH.is_file():
        raise FileNotFoundError(f"Resume checkpoint not found: {FINAL_LAST_PATH}")
    resume_model = YOLO(str(FINAL_LAST_PATH))
    resume_model.train(resume=True)